# 1. Extração e Análise Exploratória Inicial

Nesta primeira etapa do pipeline de ETL, realizamos a leitura dos dados brutos a partir do arquivo CSV e executamos a inspeção inicial da estrutura da base de dados (`shape`, `info`, `describe`).

### Objetivos:
- Validar o carregamento dos dados com a codificação (`utf-8`) e separador (`.csv`) corretos.
- Identificar o número total de registros e colunas.
- Mapear a presença de valores nulos e verificar os tipos primitivos de dados lidos pelo Pandas.

In [ ]:
import pandas as pd 
import numpy as np 

df_clientes = pd.read_csv('../data/clientes_distribuidora_vetor.csv', sep=';', encoding='utf-8') #enconding trabalha com caracteres especiais e sep trabalha com o separador do arquivo csv

display(df_clientes.head(14)) #deixa mais visualmente bonito a visualização do dataframe
display (df_clientes.shape) #mostra a quantidade de linhas e colunas do dataframe

display(df_clientes.info()) #mostra informações do dataframe, como quantidade de linhas, colunas, tipos de dados e valores nulos
display(df_clientes.describe()) #mostra estatísticas descritivas do dataframe, como média, mediana, desvio padrão, valores mínimos e máximos

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,28/02/2022,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,"Dec 24, 2023",Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,08-01-2022,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82.278.248/9638-34,+55 61 97118-8177,contato@casadastinta.com.br,Águas Claras,NaN,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83.930.103/1051-83,61999788677,contato@distribuidor.com.br,TAGUATINGA,NaN,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31.165.667/0106-51,(61) 94139-4116,contato@bazarestrela.com.br,V. Pires,72006-400,Brasília,DF,"Aug 18, 2022",Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10.801.326/7736-02,+55 61 91035-7396,contato@supermercado.com.br,Aguas claras,71926-750,Brasília,DF,2023-08-09,Juliana Reis
9,CLI0010,Papelaria Horizonte,34.309.805/0097-88,(61) 91931-9320,contato@papelariahor.com.br,Águas Claras/DF,71950-000,Brasília,DF,03/03/2021,Diego Martins


(47, 11)

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      46 non-null     str  
 6   cep                   34 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
dtypes: str(11)
memory usage: 4.2 KB


None

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
count,47,47,47,47,47,46,34,47,47,47,47
unique,47,47,47,47,44,25,15,1,1,47,5
top,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@distribuidor.com.br,Aguas claras,72140-010,Brasília,DF,2022-06-21,Camila Torres
freq,1,1,1,1,4,4,4,47,47,1,14


# 2. Transformação e Limpeza de Dados (ETL)

Nesta etapa do pipeline, tratamos as inconsistências e os ruídos encontrados na base de dados bruta da Distribuidora Vetor, preparando os campos para análises agregadas confiáveis.

### Principais Frentes de Atuação:
- **Campos Cadastrais (CNPJ e Telefones):** Remoção de caracteres especiais (pontos, barras, traços e espaços) para padronizar os dados em formato limpo.
- **Data de Cadastro:** Conversão de múltiplos formatos misturados (ISO, padrão brasileiro e texto) para o tipo data padrão, tratando falhas de conversão.
- **Normalização Textual de Bairros:** Aplicação de limpeza de texto (minúsculas, remoção de acentos e espaços extras) e estruturação de um dicionário "De-Para" para unificar grafias divergentes.
- **Enriquecimento e Tratamento de CEPs:** Identificação de valores ausentes, correção de formatos inválidos e definição de regras para resolver conflitos entre o CEP e o bairro informado.

**Resumo da etapa:** Limpeza dos caracteres especiais da coluna `cnpj` (criando a coluna `cnpjAjustado`), tratamento dos valores nulos em `bairro_informado` (substituindo por `"Não informado"`) e conversão/padronização da coluna `data_cadastro` (lidando com formatos mistos e meses em inglês) para o formato de data na cópia de trabalho (`df_analise`).

In [71]:
display(df_clientes['cnpj']) # coluna cnpj do dataframe

df_clientes['cnpjAjustado'] = df_clientes['cnpj'].str.replace(r'\D', '', regex=True) # remove caracteres não numéricos da coluna cnpj e cria uma nova coluna cnpjAjustado

df_analise = df_clientes.copy() # cria uma cópia do dataframe df_clientes para df_analise para realizar as transformações

# substitui os valores nulos da coluna bairro_informado por 'Não informado' usando o método fillna()
df_analise['bairro_informado'] = df_analise['bairro_informado'].fillna('Não informado') 

# converte datas de formatos mistos (meses em texto como dec/aug, barras e ISO) para o padrão ISO (AAAA-MM-DD) do SQLite e Django
df_analise['data_cadastro'] = pd.to_datetime(df_analise['data_cadastro'], format='mixed', dayfirst=True, errors='coerce')

display(df_analise['data_cadastro'].unique()) # exibe os valores únicos da data_cadastro para confirmar que ficaram no formato correto

display(df_analise.head(14)) # ou simplesmente escrever o nome da variável na última linha da célula: df_analise

df_analise.info() # mostra informações do dataframe df_analise para confirmar que o valor nulo foi tratado

0     84.959.310/3413-16
1     28.327.648/3503-05
2     42.388.496/9653-28
3     69.784.801/8451-46
4     89.325.288/0957-01
5     82.278.248/9638-34
6     83.930.103/1051-83
7     31.165.667/0106-51
8     10.801.326/7736-02
9     34.309.805/0097-88
10    93.990.916/9985-43
11    79.911.838/4251-35
12    24.118.244/9353-48
13        27868011280598
14    69.232.260/2563-42
15        36541458685014
16    93.406.088/3561-59
17    82.366.299/4680-44
18        48951343320037
19    20.163.287/0831-72
20    77.434.873/4714-34
21        66587603669096
22        34670656272980
23    37.556.464/1708-05
24        27193745299124
25        49190586518506
26    87.769.453/1473-79
27    49.480.831/3678-37
28    85.685.574/4431-35
29    34.352.408/2400-84
30    04.711.671/9022-94
31    74.964.990/9133-41
32    03.447.134/9361-83
33    17.464.887/7190-65
34    02.787.429/6717-56
35    07.154.516/8087-60
36    10.932.480/8613-17
37    78.263.982/1465-84
38    58.867.533/9636-05
39    71.870.262/1745-96


<DatetimeArray>
['2022-06-21 00:00:00', '2023-07-10 00:00:00', '2022-02-28 00:00:00',
 '2023-12-24 00:00:00', '2022-01-08 00:00:00', '2021-08-17 00:00:00',
 '2025-04-16 00:00:00', '2022-08-18 00:00:00', '2023-09-08 00:00:00',
 '2021-03-03 00:00:00', '2022-09-07 00:00:00', '2023-09-15 00:00:00',
 '2024-03-11 00:00:00', '2023-05-02 00:00:00', '2022-08-28 00:00:00',
 '2024-04-10 00:00:00', '2023-02-28 00:00:00', '2024-08-15 00:00:00',
 '2025-07-15 00:00:00', '2025-08-25 00:00:00', '2022-02-18 00:00:00',
 '2024-10-07 00:00:00', '2022-07-21 00:00:00', '2022-01-25 00:00:00',
 '2024-10-10 00:00:00', '2024-12-11 00:00:00', '2024-11-01 00:00:00',
 '2022-05-03 00:00:00', '2022-12-25 00:00:00', '2025-01-28 00:00:00',
 '2025-07-18 00:00:00', '2022-01-03 00:00:00', '2025-04-03 00:00:00',
 '2022-02-22 00:00:00', '2024-08-20 00:00:00', '2023-12-16 00:00:00',
 '2025-10-27 00:00:00', '2022-11-22 00:00:00', '2025-11-28 00:00:00',
 '2022-12-07 00:00:00', '2024-02-06 00:00:00', '2021-11-19 00:00:00',
 '20

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel,cnpjAjustado
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida,84959310341316
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-07-10,Camila Torres,28327648350305
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,2022-02-28,Camila Torres,42388496965328
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,2023-12-24,Diego Martins,69784801845146
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,2022-01-08,Diego Martins,89325288095701
5,CLI0006,Casa das Tintas Ipê,82.278.248/9638-34,+55 61 97118-8177,contato@casadastinta.com.br,Águas Claras,NaN,Brasília,DF,2021-08-17,Camila Torres,82278248963834
6,CLI0007,Distribuidora Popular Centro-Oeste,83.930.103/1051-83,61999788677,contato@distribuidor.com.br,TAGUATINGA,NaN,Brasília,DF,2025-04-16,Camila Torres,83930103105183
7,CLI0008,Bazar Estrela Dourada,31.165.667/0106-51,(61) 94139-4116,contato@bazarestrela.com.br,V. Pires,72006-400,Brasília,DF,2022-08-18,Juliana Reis,31165667010651
8,CLI0009,Supermercado Nova Aliança,10.801.326/7736-02,+55 61 91035-7396,contato@supermercado.com.br,Aguas claras,71926-750,Brasília,DF,2023-09-08,Juliana Reis,10801326773602
9,CLI0010,Papelaria Horizonte,34.309.805/0097-88,(61) 91931-9320,contato@papelariahor.com.br,Águas Claras/DF,71950-000,Brasília,DF,2021-03-03,Diego Martins,34309805009788


<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_cliente            47 non-null     str           
 1   razao_social          47 non-null     str           
 2   cnpj                  47 non-null     str           
 3   telefone              47 non-null     str           
 4   email                 47 non-null     str           
 5   bairro_informado      47 non-null     str           
 6   cep                   34 non-null     str           
 7   cidade                47 non-null     str           
 8   uf                    47 non-null     str           
 9   data_cadastro         47 non-null     datetime64[us]
 10  vendedor_responsavel  47 non-null     str           
 11  cnpjAjustado          47 non-null     str           
dtypes: datetime64[us](1), str(11)
memory usage: 4.5 KB
